# Clickbait: LLM level (Qwen2.5 Instruct, Kaggle GPU)
Prompting modes on the VnExpress dev headlines with `Qwen/Qwen2.5-7B-Instruct` (4-bit nf4) and, if `RUN_SMALL`,
`Qwen/Qwen2.5-1.5B-Instruct` (fp16), to compare cost and quality.

- `zero`: label definition only. `few`: 8 examples (4 per class). `cot`: the same 8 examples plus step-by-step reasoning.
- Few-shot count ablation `few<k>`, k in `FEWSHOT_K` (balanced classes; `few0` = `zero`, `few8` = `few`).
- `fewsim8` (`RUN_FEWSIM`): the 4 + 4 examples most similar to the headline (TF-IDF char n-grams).
- Examples come only from the VnExpress part of ViClickbait-2025, never from dev; ViClickbait titles that also
  appear in dev or the corpus are removed from the example pool.
- Score = P("1") / (P("0") + P("1")) for the label token (so PR-AUC can be computed); pred at 0.5. Greedy decoding.
- Final (`RUN_FINAL`): `FINAL_MODE` with the 7B model scores `corpus_headlines.csv`.

## How to run on Kaggle
1. On your machine: `python -m src.classifiers.clickbait kaggle-bundle` writes three files to `data/kaggle/clickbait/`:
   `viclickbait.csv` (id, title, sapo, source, y), `dev_headlines.csv` (id, title, sapo, y) and
   `corpus_headlines.csv` (id, title, sapo; 1,244 rows).
2. Kaggle > Datasets > New Dataset: upload the three files, set it to **Private**, e.g. `visim-clickbait`
   (the PhoBERT notebook uses the same dataset).
   Whenever the dev labels change, rerun `kaggle-bundle` and upload a new dataset version: the notebook uses the dev
   labels to pick the best configuration and as training data, so it must run on the final labels.
3. Kaggle > Code > New Notebook > File > Import Notebook: choose this `.ipynb` file.
   Settings: Accelerator **GPU T4 x2** or P100 (one GPU is used), Internet **On** (Hugging Face download, `pip install`).
   Add Input > Your Datasets: the dataset from step 2 (files are found anywhere under `/kaggle/input/`).
4. **Save Version > Save & Run All (Commit)** runs in the background; outputs are in the version's Output tab.
5. Download the outputs into `results/clickbait/kaggle/`, then run `python -m src.classifiers.clickbait collect`
   on your machine (it re-scores every file on the current dev labels).
   Record the GPU, versions, seeds and wall-clock times from the run log JSON in the task notes, with the dataset version
   (which dev labels were used).

## Estimated run time (T4, default flags)
Estimates only; actual wall-clock times are written to `qwen_runs.json`. Both notebooks together stay far below the
12-hour session limit and use about 2-3 GPU hours of the 30-hour weekly quota; if short of quota, switch off
`RUN_COT` and `RUN_SMALL` first (saves about 1-1.5 hours).

| Part | Flag | Estimate |
|---|---|---|
| Download and load Qwen2.5-7B 4-bit | always | 5-10 min |
| 7B: zero, few, few4, few16, fewsim8 | `FEWSHOT_K`, `RUN_FEWSIM` | 20-35 min |
| 7B: CoT (200 generated tokens per headline) | `RUN_COT` | 45-75 min |
| 7B: corpus scores with `FINAL_MODE` | `RUN_FINAL` | 5-10 min |
| Qwen2.5-1.5B, every mode | `RUN_SMALL` | 15-30 min |
| **Total** | | **about 1.5-2.5 hours** |

## Output files (`/kaggle/working`, download into `results/clickbait/kaggle/`)
All score files have columns `id, pred, score` (score = P(clickbait), 4 decimals).
- `dev_scores_qwen_zero.csv`, `_few.csv`, `_cot.csv`, `_few0/_few4/_few8/_few16.csv` (`few0` = `zero`, `few8` = `few`),
  `_fewsim8.csv`; the 1.5B files use the prefix `qwen15_`.
- `qwen_cot_outputs.jsonl`, `qwen15_cot_outputs.jsonl`: CoT reasoning per dev item (for error analysis).
- `corpus_scores_qwen.csv`: corpus scores of the 7B model in `FINAL_MODE`.
- `qwen_runs.json`: GPU, versions, prompt, time per mode and per model, dev metrics per run.


In [ ]:
!pip install -q -U bitsandbytes accelerate

In [ ]:
# ---- Configuration: switch expensive parts off here ----
MODELS = [("Qwen/Qwen2.5-7B-Instruct", "qwen", True)]        # (Hugging Face id, config prefix, 4-bit)
RUN_SMALL = True                                             # also Qwen2.5-1.5B-Instruct (fp16), prefix qwen15
if RUN_SMALL:
    MODELS.append(("Qwen/Qwen2.5-1.5B-Instruct", "qwen15", False))
FEWSHOT_K = [0, 4, 8, 16]    # few-shot count ablation (0 and 8 reuse the zero and few runs)
RUN_FEWSIM = True            # similarity-selected examples (fewsim8)
RUN_COT = True               # Chain-of-Thought (generation, the slowest mode)
RUN_FINAL = True             # score corpus_headlines.csv
FINAL_MODE, FINAL_PREFIX = "few", "qwen"
SEED = 42
BATCH, COT_BATCH, COT_MAX_NEW_TOKENS = 8, 8, 200

In [ ]:
import gc, glob, json, os, re, shutil, time
import numpy as np, pandas as pd, torch, transformers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import average_precision_score, f1_score
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

try:
    import bitsandbytes
    BNB = bitsandbytes.__version__
except Exception:
    BNB = None


def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    return hits[0] if hits else None


assert find("viclickbait.csv"), "Add the Kaggle dataset built by `kaggle-bundle` (Add Input)"
OUT = "/kaggle/working"
torch.manual_seed(SEED)


def read(name):
    df = pd.read_csv(find(name), dtype={"id": str})
    df["title"] = df["title"].fillna("").astype(str)
    return df


dev = read("dev_headlines.csv")
vc = read("viclickbait.csv")
corpus = read("corpus_headlines.csv") if find("corpus_headlines.csv") else None


def norm(t):
    return re.sub(r"\s+", " ", t).strip().lower()


seen = set(dev["title"].map(norm)) | (set(corpus["title"].map(norm)) if corpus is not None else set())
vne = vc[vc["source"] == "VnExpress"]
pool = vne[~vne["title"].map(norm).isin(seen)].reset_index(drop=True)

ENV = {"gpu": torch.cuda.get_device_name(0), "gpu_count": torch.cuda.device_count(), "torch": torch.__version__,
       "cuda": torch.version.cuda, "transformers": transformers.__version__, "bitsandbytes": BNB}
print(ENV)
print(f"dev {len(dev)} (positive {dev['y'].mean():.3f}); corpus {None if corpus is None else len(corpus)}; "
      f"example pool {len(pool)} VnExpress ViClickbait titles ({len(vne) - len(pool)} removed as overlapping)")
if corpus is None:
    print("WARNING: corpus_headlines.csv not found; RUN_FINAL will be skipped")

In [ ]:
# Label definition: the clickbait definition in TASK.md
DEFINITION = """Bạn là người gán nhãn tiêu đề báo tiếng Việt. Tiêu đề là clickbait (1) nếu có ít nhất một dấu hiệu:
1. Cố ý giấu thông tin chính để kích thích tò mò: đại từ mơ hồ ("người này", "điều đó", "lý do..."); câu hỏi gợi tò mò mà tiêu đề không trả lời ("Vì sao...?", "Điều gì...?"); danh sách không nói rõ nội dung; biệt danh thay cho tên thật.
2. Từ ngữ cảm xúc, cường điệu do người viết chọn để gây ấn tượng ("gây sốc", "chấn động", "gây bão", ẩn dụ giật gân).
3. Bỏ sót chi tiết then chốt khiến người đọc dễ hiểu sai.
Không clickbait (0): tiêu đề đủ thông tin, khách quan. Từ mạnh nhưng mô tả sự kiện ("bắt", "tử vong", "khởi tố") không phải cường điệu. Trích dẫn có nêu người nói, tên tác phẩm trong ngoặc kép, câu hỏi chính sách hoặc hướng dẫn thật ("Có nên...?", "Làm ... thế nào?") đều là 0."""

# mode -> (number of examples, similarity-selected, chain of thought)
MODE_SPEC = {"zero": (0, False, False), "few": (8, False, False)}
for k in FEWSHOT_K:
    if k not in (0, 8):
        MODE_SPEC[f"few{k}"] = (k, False, False)
if RUN_FEWSIM:
    MODE_SPEC["fewsim8"] = (8, True, False)
if RUN_COT:
    MODE_SPEC["cot"] = (8, False, True)
MODES = list(MODE_SPEC)
COPIES = {"zero": "few0" if 0 in FEWSHOT_K else None, "few": "few8" if 8 in FEWSHOT_K else None}


def fixed_shots(k):
    """k/2 positive and k/2 negative VnExpress ViClickbait headlines (k=8 gives the original `few` set)."""
    pos = pool[pool["y"] == 1].sample(k // 2, random_state=SEED)
    neg = pool[pool["y"] == 0].sample(k // 2, random_state=SEED)
    return pd.concat([pos, neg]).sample(frac=1, random_state=SEED)


FIXED = {k: fixed_shots(k) for k, sim, _ in MODE_SPEC.values() if k and not sim}
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True).fit(pool["title"])
P = vec.transform(pool["title"])
POOL_Y = pool["y"].to_numpy()


def similar_shots(title, k):
    sim = (P @ vec.transform([title]).T).toarray().ravel()
    pick = []
    for y in (1, 0):
        idx = np.where(POOL_Y == y)[0]
        pick += list(idx[np.argsort(-sim[idx], kind="stable")[: k // 2]])
    return pool.iloc[sorted(pick, key=lambda i: sim[i])]  # most similar example last, next to the headline


def messages(title, mode):
    k, sim, cot = MODE_SPEC[mode]
    task = DEFINITION
    if k:
        shots = similar_shots(title, k) if sim else FIXED[k]
        task += "\n\nVí dụ:\n" + "\n".join(f"Tiêu đề: {r.title}\nNhãn: {r.y}" for r in shots.itertuples())
    if cot:
        task += "\n\nHãy xét lần lượt ba dấu hiệu trong 2 đến 3 câu ngắn, rồi kết thúc bằng một dòng 'Nhãn: 0' hoặc 'Nhãn: 1'."
        user = f"Tiêu đề: {title}"
    else:
        task += "\n\nChỉ trả lời một chữ số 0 hoặc 1."
        user = f"Tiêu đề: {title}\nNhãn:"
    return [{"role": "system", "content": task}, {"role": "user", "content": user}]


print(json.dumps(messages(dev["title"].iloc[0], "few"), ensure_ascii=False, indent=1)[:3000])

In [ ]:
def load(model_id, four_bit):
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = {"device_map": {"": 0}, "torch_dtype": torch.float16}
    if four_bit:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                                       bnb_4bit_quant_type="nf4")
    model = AutoModelForCausalLM.from_pretrained(model_id, **kw).eval()
    return tok, model


def prompt(title, mode):
    return tok.apply_chat_template(messages(title, mode), tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def label_probs(prompts):
    """P("1") / (P("0") + P("1")) for the next token, last position only (no full-vocabulary logits per position)."""
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    pos = (enc["attention_mask"].cumsum(-1) - 1).clamp(min=0)
    h = model.model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], position_ids=pos).last_hidden_state[:, -1]
    logits = model.lm_head(h.to(model.lm_head.weight.device))
    return torch.softmax(logits[:, [ID0, ID1]].float(), -1)[:, 1].cpu().numpy().tolist()


@torch.no_grad()
def generate(prompts):
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    gen = model.generate(**enc, max_new_tokens=COT_MAX_NEW_TOKENS, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)


def batched(items, fn, bs, desc):
    """Run fn over items in length-sorted batches (less padding); results in the original order."""
    order = np.argsort([len(x) for x in items], kind="stable")
    out = [None] * len(items)
    for i in tqdm(range(0, len(order), bs), desc=desc):
        idx = order[i:i + bs]
        for j, r in zip(idx, fn([items[j] for j in idx])):
            out[j] = r
    return out


def score_titles(titles, mode, desc):
    prompts = [prompt(t, mode) for t in titles]
    if not MODE_SPEC[mode][2]:
        return np.array(batched(prompts, label_probs, BATCH, desc)), None
    texts = batched(prompts, generate, COT_BATCH, desc + " (generate)")
    second = [p + t.split("Nhãn:")[0].rstrip() + "\nNhãn: " for p, t in zip(prompts, texts)]
    return np.array(batched(second, label_probs, BATCH, desc + " (label)")), texts


def write_scores(path, ids, s):
    pd.DataFrame({"id": ids, "pred": (s >= 0.5).astype(int), "score": np.round(s, 4)}).to_csv(path, index=False)


def metrics(y, s):
    pred = (s >= 0.5).astype(int)
    return {"f1": float(f1_score(y, pred, zero_division=0)), "pr_auc": float(average_precision_score(y, s)),
            "predicted_positive_share": float(pred.mean())}


LOG = {"env": ENV, "seed": SEED, "definition": DEFINITION, "modes": {m: dict(zip(("k", "similar", "cot"), v)) for m, v in MODE_SPEC.items()},
       "example_pool": {"source": "ViClickbait-2025, VnExpress part, titles overlapping dev/corpus removed", "n": len(pool),
                        "positive": int(pool["y"].sum())},
       "batch": BATCH, "cot_batch": COT_BATCH, "cot_max_new_tokens": COT_MAX_NEW_TOKENS, "models": [], "runs": [], "final": None}


def save_log():
    with open(f"{OUT}/qwen_runs.json", "w") as f:
        json.dump(LOG, f, indent=2, ensure_ascii=False, default=float)

In [ ]:
T_START = time.time()
for model_id, prefix, four_bit in MODELS:
    t0 = time.time()
    tok, model = load(model_id, four_bit)
    ID0, ID1 = tok.encode("0", add_special_tokens=False)[0], tok.encode("1", add_special_tokens=False)[0]
    LOG["models"].append({"model": model_id, "prefix": prefix, "quantization": "4-bit nf4, fp16 compute" if four_bit else "fp16",
                          "load_seconds": round(time.time() - t0, 1),
                          "gpu_memory_gb": round(torch.cuda.memory_allocated(0) / 1e9, 2)})
    print(LOG["models"][-1])
    for mode in MODES:
        t = time.time()
        name = f"{prefix}_{mode}"
        s, texts = score_titles(dev["title"].tolist(), mode, name)
        write_scores(f"{OUT}/dev_scores_{name}.csv", dev["id"], s)
        if COPIES.get(mode):
            shutil.copy(f"{OUT}/dev_scores_{name}.csv", f"{OUT}/dev_scores_{prefix}_{COPIES[mode]}.csv")
        run = {"config": name, "model": model_id, "mode": mode, **LOG["modes"][mode], "n": len(dev),
               "seconds": round(time.time() - t, 1), **metrics(dev["y"].to_numpy(), s)}
        if texts is not None:
            parsed = [re.search(r"Nhãn:\s*([01])", x) for x in texts]
            run["cot_unparsed_share"] = float(np.mean([p is None for p in parsed]))
            with open(f"{OUT}/{prefix}_cot_outputs.jsonl", "w") as f:
                for i, x, p, sc in zip(dev["id"], texts, parsed, s):
                    f.write(json.dumps({"id": i, "output": x, "generated_label": int(p.group(1)) if p else None,
                                        "score": round(float(sc), 4)}, ensure_ascii=False) + "\n")
        LOG["runs"].append(run); save_log()
        print({k: round(v, 3) if isinstance(v, float) else v for k, v in run.items()})
    if RUN_FINAL and prefix == FINAL_PREFIX and corpus is not None:
        t = time.time()
        s, texts = score_titles(corpus["title"].tolist(), FINAL_MODE, f"corpus {prefix}_{FINAL_MODE}")
        write_scores(f"{OUT}/corpus_scores_{prefix}.csv", corpus["id"], s)
        if texts is not None:
            with open(f"{OUT}/corpus_{prefix}_cot_outputs.jsonl", "w") as f:
                for i, x in zip(corpus["id"], texts):
                    f.write(json.dumps({"id": i, "output": x}, ensure_ascii=False) + "\n")
        LOG["final"] = {"config": f"{prefix}_{FINAL_MODE}", "model": model_id, "file": f"corpus_scores_{prefix}.csv",
                        "n": len(corpus), "seconds": round(time.time() - t, 1),
                        "predicted_positive_share": float((s >= 0.5).mean())}
        save_log(); print(LOG["final"])
    del model, tok
    gc.collect(); torch.cuda.empty_cache()

LOG["total_seconds"] = round(time.time() - T_START, 1)
save_log()
print(f"total {LOG['total_seconds'] / 60:.1f} min on {ENV['gpu']}")
pd.DataFrame(LOG["runs"])[["config", "f1", "pr_auc", "predicted_positive_share", "seconds"]].round(3)